# Validação de Precificação de TPFs com Base na ANBIMA (Mercado Secundário)
Este notebook importa o arquivo de PUs oficiais da ANBIMA e bate os preços calculados pela nossa engine em Cython contra eles.

In [1]:
import datetime
import pandas as pd
import numpy as np
from bz_pricing.data.loaders import B3DataLoader
from bz_pricing.curves.builder import B3CurveBuilder
from bz_pricing.core.curves import FlatCurve
from bz_pricing.instruments.tpf import LTN, NTNF, LFT, NTNB, NTNC

base_date = datetime.date(2026, 9, 4)
data_dir = "../dados/b3"

loader = B3DataLoader(data_dir)
builder = B3CurveBuilder()
cal = builder.cal

## Carrega ANBIMA MS

In [2]:
# Carrega arquivo MS que já baixamos com o scraper
df_ms = loader.load_anbima_ms(base_date)
df_ms.head()

,Titulo,Data Referencia,CodigoSelic,DataEmissao,DataVencimento,Tx. Compra,Tx. Venda,TaxaIndicativa,PU,Desvio padrao,Interv. Ind. Inf. (D0),Interv. Ind. Sup. (D0),Interv. Ind. Inf. (D+1),Interv. Ind. Sup. (D+1),Criterio
0,LTN,2026-09-04,100000,2024-07-05,2026-10-01,13.7753,13.7301,13.7613,990.832833,0.016433,13.5879,13.9036,13.6125,13.9329,Calculado
1,LTN,2026-09-04,100000,2025-01-10,2027-04-01,13.3757,13.3411,13.3541,932.732463,0.012184,13.1320,13.6311,13.1419,13.6380,Calculado
2,LTN,2026-09-04,100000,2023-07-07,2027-07-01,13.2839,13.2554,13.2722,904.483495,0.003437,12.9935,13.6373,12.9942,13.6363,Calculado
3,LTN,2026-09-04,100000,2025-07-04,2027-10-01,13.5140,13.4997,13.5066,873.947774,0.003984,13.1709,13.9202,13.1719,13.9198,Calculado
4,LTN,2026-09-04,100000,2024-01-05,2028-01-01,13.6075,13.5971,13.6030,845.757654,0.000000,13.2300,14.0549,13.2187,14.0426,Calculado


## Validação das LTNs

In [3]:
df_ltn = df_ms[df_ms['Titulo'] == 'LTN'].copy()

resultados_ltn = []

for idx, row in df_ltn.iterrows():
    venc = row['DataVencimento'].date()
    rate = row['TaxaIndicativa'] / 100.0
    pu_oficial = row['PU']
    
    # Cria curva Flat com a Taxa exata do MS
    flat_curve = FlatCurve(rate)
    
    # Constrói o Título no Cython com regras ANBIMA
    ltn = LTN(maturity_date=venc)
    
    # Calcula PU
    pu_calc = ltn.present_value(base_date, flat_curve)
    diff = abs(pu_calc - pu_oficial)
    
    # Inverso: DU implícito na ANBIMA vs DU nosso
    du_nosso = cal.bizdays(base_date, ltn.maturity_date)
    du_anbima = np.round(252.0 * np.log(1000.0 / pu_oficial) / np.log(1.0 + rate))
    
    resultados_ltn.append({
        'Vencimento': venc,
        'Taxa': rate,
        'PU Oficial': pu_oficial,
        'PU Calc': pu_calc,
        'Dif': diff,
        'DU ANBIMA': int(du_anbima),
        'DU Nosso': du_nosso
    })

df_res_ltn = pd.DataFrame(resultados_ltn)
df_res_ltn

,Vencimento,Taxa,PU Oficial,PU Calc,Dif,DU ANBIMA,DU Nosso
0,2026-10-01,0.137613,990.832833,990.832833,0.0,18,18
1,2027-04-01,0.133541,932.732463,932.732463,0.0,140,140
2,2027-07-01,0.132722,904.483495,904.483495,0.0,203,203
3,2027-10-01,0.135066,873.947774,873.947774,0.0,268,268
4,2028-01-01,0.136030,845.757654,845.757654,0.0,331,331
5,2028-04-01,0.136750,818.404996,818.404996,0.0,394,394
6,2028-07-01,0.137050,793.025040,793.025040,0.0,455,455
7,2028-10-01,0.138475,765.597337,765.597337,0.0,519,519
8,2029-01-01,0.139150,741.307874,741.307874,0.0,579,579
9,2029-07-01,0.140400,693.152032,693.152032,0.0,703,703


In [4]:
print("--- NTN-F ---")
df_ntnf = df_ms[df_ms['Titulo'] == 'NTN-F'].copy()
resultados_ntnf = []

for idx, row in df_ntnf.iterrows():
    venc = row['DataVencimento'].date()
    rate = row['TaxaIndicativa'] / 100.0
    pu_oficial = row['PU']
    
    flat_curve = FlatCurve(rate)
    ntnf = NTNF(maturity_date=venc, coupon_rate=0.10)
    pu_calc = ntnf.present_value(base_date, flat_curve)
    diff = abs(pu_calc - pu_oficial)
    
    resultados_ntnf.append({
        'Vencimento': venc,
        'Taxa': rate,
        'PU Oficial': pu_oficial,
        'PU Calc': pu_calc,
        'Dif': diff
    })

df_res_ntnf = pd.DataFrame(resultados_ntnf)
df_res_ntnf

--- NTN-F ---


,Vencimento,Taxa,PU Oficial,PU Calc,Dif
0,2027-01-01,0.133751,1007.834317,1007.834317,0.0
1,2029-01-01,0.137226,951.297456,951.297456,0.0
2,2031-01-01,0.141985,894.577656,894.577656,0.0
3,2033-01-01,0.143177,852.433375,852.433375,0.0
4,2035-01-01,0.143422,822.624238,822.624238,0.0
5,2037-01-01,0.143305,800.512894,800.512894,0.0
